# 수정 분석 결과 보고서 생성

수정 분석의 실제 실행 산출물을 읽어 연구 결과와 해석을 정리한다. 결과를 보기 전에 저장한 분석 계획과 원자료 해시를 확인한다. 이 노트북은 새 회귀를 적합하지 않고, 검토한 표와 서술을 하나의 읽기용 HTML로 작성한다. 기존 보고서는 보존한다.

In [1]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import json, hashlib, shutil, subprocess, sqlite3
import pandas as pd
import numpy as np
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'AGENTS.md').exists() and (p/'final').is_dir())
OUT=ROOT/'reports/final/revised_quarterly'
sha=lambda p:hashlib.sha256(p.read_bytes()).hexdigest()
load=lambda p:json.loads(p.read_text(encoding='utf-8'))
PLAN=ROOT/'final/metadata/revised_quarterly_plan.json'
plan=load(PLAN)
data_summary=load(OUT/'12_dataset_summary.json')
analysis_summary=load(OUT/'13_analysis_summary.json')
selection_summary=load(OUT/'14_summary.json')
assert data_summary['validation_all_passed'] and data_summary['all_inputs_unchanged']
for name,digest in plan['protected_source_sha256'].items():assert sha(ROOT/name)==digest,name
paths={k:OUT/name for k,name in {
'effects':'13_effects.csv','spatial':'13_spatial_se_sensitivity.csv',
'comparison':'13_exposure_comparison.csv','information':'13_information_concentration.csv',
'zero':'13_zero_night_diagnostics.csv','common10':'13_common10_support.csv',
'implementation':'13_implementation_checks.csv','availability':'12_sample_availability.csv'}.items()}
frames={key:pd.read_csv(path) for key,path in paths.items()}
input_hashes={path.relative_to(ROOT).as_posix():sha(path) for path in [PLAN,*paths.values(),OUT/'12_dataset_summary.json',OUT/'13_analysis_summary.json',OUT/'14_summary.json']}
effects=frames['effects']
PRIMARY='primary_centroid100';TOTAL='log1p_total_sales_won';NIGHT='log1p_night_sales_21_24_won'
main=effects.loc[effects.scenario.eq(PRIMARY)].copy()
assert len(main)==20 and len(effects)==80
print('기본 분석 20개, 고정된 비교 분석 포함 80개 결과 행 로드')

기본 분석 20개, 고정된 비교 분석 포함 80개 결과 행 로드


## 1. 표·차트 자료와 출처 만들기

해당 CSV를 바탕으로 표시용 열을 만든 후 SQLite에서 실제 조회한 값을 artifact에 저장한다. 표의 신뢰구간은 개별 95% 구간이며, 다중비교 보정 구간이 아니다. 미추정 행을 표에서 제거하지 않는다.

In [2]:
connection=sqlite3.connect(':memory:')
datasets={};sources=[];blocks=[];tables=[];charts=[]
GENERATOR='final/report/08_수정분석_결과보고서.ipynb'
def dataset(key,frame,path,description):
    frame.to_sql(key,connection,index=False,if_exists='replace')
    query='SELECT * FROM "'+key+'"'
    result=pd.read_sql_query(query,connection)
    datasets[key]=json.loads(result.to_json(orient='records',force_ascii=False))
    sources.append({'id':key,'label':description,'path':path.relative_to(ROOT).as_posix(),
        'query':{'engine':'sqlite','language':'sql','sql':query,'description':description,
        'tables_used':[key],'files_used':[path.relative_to(ROOT).as_posix(),GENERATOR]}})
def md(key,body,source=None):
    item={'id':key,'type':'markdown','body':body}
    if source:item['sourceId']=source
    blocks.append(item)
def table(key,title,columns,sort):
    tables.append({'id':key,'title':title,'dataset':key,'sourceId':key,
        'defaultSort':{'field':sort,'direction':'asc'},
        'columns':[{'field':f,'label':label,'type':kind,**({'format':'number'} if kind=='number' else {})} for f,label,kind in columns]})
    blocks.append({'id':key+'_block','type':'table','tableId':key})
def bar(key,title,category,value,label,unit,subtitle):
    charts.append({'id':key+'_chart','title':title,'dataset':key,'sourceId':key,'type':'horizontalBar',
        'subtitle':subtitle,'showDescription':True,'valueFormat':'number','unit':unit,
        'encodings':{'x':{'field':category,'type':'nominal','label':'업종'},'y':{'field':value,'type':'quantitative','label':label}}})
    blocks.append({'id':key+'_chart_block','type':'chart','chartId':key+'_chart'})
def interval(row,prefix='doubling'):
    if row['status']!='estimated':return '미추정'
    return f"{row[prefix+'_pct']:+.2f}% [{row[prefix+'_ci_low']:+.2f}%, {row[prefix+'_ci_high']:+.2f}%]"
def probability(value):return '—' if pd.isna(value) else f'{value:.4f}'
status_labels={'estimated':'추정','too_few_clusters':'20상권 미만','no_identifiable_within_change':'식별 변동 부족'}
for outcome,key in [(TOTAL,'total'),(NIGHT,'night')]:
    part=main.loc[main.outcome.eq(outcome)].copy()
    part['effect_ci']=part.apply(interval,axis=1)
    for field in ['p_value','p_bh10','p_bh20']:part[field+'_display']=part[field].map(probability)
    part['status_label']=part.status.map(status_labels).fillna(part.status)
    dataset(key,part,paths['effects'],'기본 중심점500m·일정 가용률 100%·예정 시작회차. 1+노출 2배에 대한 매출+1 조건부 기하평균 상대차이. CI는 상권군집 CR1 및 t(G-1)의 개별 95%.')
for key,name in [('plan','revised_quarterly_plan.json'),('data_summary','12_dataset_summary.json'),('analysis_summary','13_analysis_summary.json'),('selection_summary','14_summary.json')]:
    path=PLAN if key=='plan' else OUT/name
    sources.append({'id':key,'label':name,'path':path.relative_to(ROOT).as_posix()})
print('기본 추정:',main.status.value_counts().to_dict())

기본 추정: {'estimated': 20}


In [3]:
scenario_labels={'primary_centroid100':'기본: 중심점500m·100%','centroid90':'중심점500m·90%','min_distance100':'경계 최단거리500m·100%','id_same_sample':'기본과 동일 표본·공연 ID 수'}
outcome_labels={TOTAL:'총매출액',NIGHT:'21~24시 매출액'}
scenario_rows=[]
for (scenario,outcome),part in effects.groupby(['scenario','outcome']):
    fit=part.loc[part.status.eq('estimated')]
    scenario_rows.append({'scenario':scenario_labels[scenario],'outcome':outcome_labels[outcome],
        'estimated':len(fit),'raw_significant':int(fit.p_value.lt(.05).sum()),
        'bh10_significant':int(fit.p_bh10.lt(.05).sum()),'bh20_significant':int(fit.p_bh20.lt(.05).sum()),
        'rows':int(fit.nobs_total.iloc[0]) if len(fit) else 0,
        'clusters':int(fit.clusters_total.iloc[0]) if len(fit) else 0})
scenario_table=pd.DataFrame(scenario_rows)
dataset('scenarios',scenario_table,paths['effects'],'사전에 고정한4사양×2결과. 검정군은 결과별10개, 사양별20개. 미추정은 보정 계산에서 p=1로 유지.')
wide=[]
for industry,part in effects.loc[effects.outcome.eq(TOTAL)].groupby('industry'):
    row={'industry':industry}
    for scenario in scenario_labels:
        v=part.loc[part.scenario.eq(scenario)].iloc[0]
        row[scenario]=interval(v)
    wide.append(row)
dataset('total_sensitivity',pd.DataFrame(wide),paths['effects'],'총매출의4사양별 효과·개별 95%CI. 예정회차와 ID수는 기본과 동일행 비교이나 단위는 다름. 공간·가용률 비교는 표본도 달라짐.')
support=main.loc[main.outcome.eq(TOTAL),['industry','nobs_industry','clusters_industry','status']].copy()
dataset('support',support,paths['effects'],'기본 분석의 업종별 상권 수. 20상권 미만은 추정하지 않으며 업종 간 동일한 지역 표본을 뜻하지 않음.')
spatial=frames['spatial'].copy()
spatial['outcome_label']=spatial.outcome.map(outcome_labels)
spatial['spatial_ci']=spatial.apply(lambda r:f"{r.doubling_spatial_ci_low:+.2f}% ~ {r.doubling_spatial_ci_high:+.2f}%",axis=1)
dataset('spatial',spatial,paths['spatial'],'기본 분석의 같은계수·표본. 상권별 누적점수에 exp[-(거리/1000m)^2] 가중 공분산. 참고용정규95%구간이며 p값이나 정확한 유한표본 추론으로 해석하지 않음.')
assert frames['comparison'].identical_candidate_rows.all()
assert frames['comparison'].identical_entire_estimated_sample.all()
print('동일 표본 노출 비교 확인:',len(frames['comparison']))

동일 표본 노출 비교 확인: 20


## 2. 수정 이유와 현재 측정 대상

이전 진단의 지적 사항과 실제 변경 내용을 연결한다. 분기 관측을 정밀한 공연 후 매출로 바꾸지 않는다. 아래 표시용 표와 차트의 수치는 실행 산출물에서만 가져온다.

In [4]:
title='공연 예정회차와 주변 외식업 매출: 수정 분석 결과'
md('title','# '+title+chr(10)+chr(10)+'**학부 졸업 프로젝트 · 분석 수정 및 재실행**  '+chr(10)+'종로구·마포구 · 외식업 10종 · 2023~2025년 12분기')
# 실제 추정 결과를 확인한 요약은 뒤에서 title 다음 위치에 삽입한다.
md('definitions',"""## 공연 ID 수·야간 비중에서 예정회차·매출액으로 질문을 구체화했다

이번 질문은 **같은 상권·업종에서 예정 공연량이 달라진 분기에 매출액이 어떻게 달라지는가**다. 주 결과는 분기 총매출액, 보조 결과는 그 분기의 모든 21~24시 매출을 합한 금액이다. 공연 종료 직후 3시간이나 6시간의 매출을 관측한 결과가 아니다.

공연 활동은 **상권 중심점에서 500m 이내 시설의 예정 시작회차 수**다. 등록기간에 상세정보의 일반 요일·시각을 반복해 만들었다. 실제 회차나 관객 수를 새로 확보한 것이 아니며, 휴연·과거 일정 변경·휴일 특수 일정은 완전히 반영하지 못한다.

기본 표본은 해당 상권·분기의 공식 연결 후보 공연 ID 모두에 대해 정상 요일 시작일정을 재구성할 수 있는 경우다. **가용률 100%는 후보 ID 중 시작일정 가용 기준을 충족하는 비율**이다. 일정 파싱·등록기간이 유효하고 해당 분기에 시작회차가 하나 이상 생성되어야 하며, 정상적으로 해석된 러닝타임으로 볼 때 같은 ID의 회차가 서로 겹치는 후보는 제외한다. 이는 실제 일정 정확도나 실제 관객의 관측률 100%를 뜻하지 않는다. 등록 후보가 없는 경우의 노출 0도 미등록 공연까지 없었다는 증거가 아니다.""")
md('changes',"""수정은 앞선 진단에서 확인한 문제에 대응한다.

- 장기 공연과 단기 공연을 모두 1개로 세던 차이를 줄이기 위해 **공연 ID 수를 예정 시작회차로 바꾸었다**. 시작 횟수에는 러닝타임 결측을 필수 제외 사유로 삼지 않았다.
- 공연장과 상권 경계의 일부만 가까웠던 연결을 점검하기 위해 **중심점 500m를 기본**으로 정하고 기존 최단거리 연결을 비교에 남겼다. 전체 상권 매출을 좁은 원 안의 매출로 환산하지 않았다.
- 총매출 증가와 야간 비중 증가를 혼동하지 않도록 **총매출액·야간 매출액을 각각 분석**했다. 비중은 주 회귀 결과에서 제외했다.
- 분기 전체의 공통 매출 변화를 제거하던 시간대 모형 대신 **상권×업종, 업종×분기 고정효과**를 사용했다. 같은 상권의 변화와 업종별 공통 분기 변화를 구분한다.
- 가까운 상권 사이의 상관 가능성을 고려한 **공간 표준오차 민감도**를 추가했다. 통계적 유의성을 얻기 위해 기준이나 표본을 사후 변경하지 않았다.""")
md('method',"""## 분석 모형과 표를 읽는 방법

**ln(1+매출ₐᵢq) = 상권×업종 고정효과 + 업종×분기 고정효과 + βᵢ ln(1+예정회차ₐq) + 오차**

매출이 제공되고 총매출이 양수이며 야간 매출이 0 이상인 공통 표본을 쓴다. 동일 상권×업종에 2분기 이상 관측을 요구하고, 업종별 20개 상권 미만은 추정하지 않는다. 미제공 매출은 0원으로 채우지 않았으며 실제로 제공된 야간 0원은 보존했다.

아래 상대차이는 **1+예정회차가 2배일 때 매출+1의 조건부 기하평균 차이**다. 예를 들어 9회에서 19회, 99회에서 199회가 이 비교에 해당한다. 원화 평균매출의 인과효과나 단순히 공연 수 자체를 두 배 늘렸을 때의 효과가 아니다.

기본 표준오차는 상권별 군집 CR1, 개별 95% 신뢰구간은 t(G−1) 기준이다. BH10은 결과별 10업종, BH20은 총매출·야간 매출을 합친 20개 검정의 다중비교 보정 p값이다. 미추정 결과도 보정 계산에서는 p=1로 유지했다. 신뢰구간 자체는 동시 보정 구간이 아니다.""")
md('support_heading','## 업종별로 실제 비교에 포함되는 상권이 다르다'+chr(10)+chr(10)+'아래는 기본 분석에서 업종별로 남은 상권 수다. 같은 10종을 대상으로 해도 매출 제공 여부와 일정 가용성에 따라 관측 지역이 다르다. 20상권 미만인 업종은 빈 결과로 남긴다. 상권 수는 독립적인 공연 충격의 개수와 같지 않다.')
bar('support','기본 분석의 업종별 상권 수','industry','clusters_industry','상권 수','개','2023~2025년 · 중심점500m · 일정 가용률 100% · 최소20상권 기준')
md('total_heading','## 총매출액: 효과 크기와 불확실성을 함께 본다'+chr(10)+chr(10)+'양수는 예정회차가 많은 분기에 총매출도 높은 조건부 관련성, 음수는 반대 관련성을 뜻한다. 계수의 방향만으로 결론내리지 않고 개별 구간과 보정 p값을 함께 읽는다. 업종 간 차이는 별도의 계수 차이 검정 결과가 아니다.')
columns=[('industry','업종','text'),('effect_ci','상대차이 [개별 95% CI]','text'),('p_value_display','p','text'),('p_bh10_display','BH10','text'),('p_bh20_display','BH20','text'),('clusters_industry','상권 수','number'),('status_label','추정 상태','text')]
table('total','예정회차와 분기 총매출액',columns,'industry')
md('night_heading','## 야간 매출액은 야간 비중과 다른 결과다'+chr(10)+chr(10)+'야간 매출액은 21~24시의 분기 합계다. 총매출이 더 빠르게 늘면 야간 매출액이 늘어도 야간 비중은 낮아질 수 있다. 따라서 이전 야간 비중 계수와 아래 금액 계수의 크기를 직접 비교하지 않는다. 제공된 0원을 유지한 ln(1+금액) 결과에는 0원과 양수 사이의 전환도 포함된다.')
table('night','예정회차와 분기 21~24시 매출액',columns,'industry')

## 3. 대표성·민감도·해석 한계

공연 밀집 지역의 선택을 먼저 보여준 뒤 회귀 결과를 읽도록 배치한다. 가용률 기준을 완화하는 비교는 미확인 일정을 해결하지 못하므로 불완전 부분 합계라는 한계를 함께 설명한다.

In [5]:
density_path=OUT/'14_candidate_density_bands.csv'
area_selection_path=OUT/'14_area_selection.csv'
density=pd.read_csv(density_path)
area_selection=pd.read_csv(area_selection_path)
for path in [density_path,area_selection_path]:input_hashes[path.relative_to(ROOT).as_posix()]=sha(path)
dataset('density',density,density_path,'중심점500m에서 파싱전후보ID수 구간별, 제공매출행 중 일정 가용률 100%를 통과한 비율. 반복관측·최소군집 필터 전.')
zero_selection=next(x for x in selection_summary['selection'] if x['sample']=='repeat_ge2_before_min_cluster_filter')
high_density=density.loc[density.candidate_density_band.eq('100+')].iloc[0]
hubs=area_selection.loc[area_selection.area_name.isin(['대학로(혜화역)','홍대입구역(홍대)'])]
assert len(hubs)==2 and hubs.retained_sales_rows.eq(0).all()
prior_block_count=len(blocks)
md('selection_heading',f"""## 엄격한 일정 기준은 핵심 공연 상권을 많이 제외한다

매출이 제공된 7,661행 중 일정 가용률 100% 기준을 통과한 것은 5,042행이다. 반복 관측 조건 후에는 {zero_selection['sales_rows']:,}행·{zero_selection['areas']}상권이 남는다. 이 중 {zero_selection['zero_candidate_sales_row_pct']:.1f}%는 등록 후보 공연이 없는 관측이며, 선택된 기간 내내 노출0인 상권×업종에 속한 행도 {zero_selection['sales_rows_in_all_zero_entities_pct']:.1f}%다. 0노출 관측은 공통 분기 변화를 비교하는 데 기여할 수 있으므로 모두 무용한 행이라고 볼 수는 없다.

**대학로(혜화역)와 홍대입구역(홍대)은 각각 제공된 120행이 기본 표본에서 모두 제외됐다.** 후보 공연이 분기 100개 이상인 집단은 제공된 {int(high_density.provided_sales_rows):,}행 중 {int(high_density.retained_sales_rows)}행({high_density.retained_of_provided_sales_rows_pct:.2f}%)만 남는다. 아래 비율은 최소군집 조건을 적용하기 전의 선택 단계다. 가용률 100% 표본은 공연 밀집 중심상권의 대표 표본으로 해석할 수 없다.""",'selection_summary')
bar('density','공연 밀도별 제공 매출행의 유지율','candidate_density_band','retained_of_provided_sales_rows_pct','제공행 중 유지율 (%)','%','파싱 전 분기 후보 공연ID수 기준 · 일정 가용률 100% · 반복관측·최소군집 필터 전')
charts[-1]['encodings']['x']['label']='분기 후보 공연ID수'
md('selection_meaning',"""위 시작일정 가용 기준을 충족하지 못한 후보 ID가 하나라도 있으면 해당 상권·분기를 제외하는 방식은 부분 회차 합계를 피하는 대신 공연이 많은 상권을 더 자주 제외한다. 따라서 이번 기본 분석의 의미는 **일정 재구성이 가능한 표본에서의 관련성**이다. 90% 표본을 추가해 이 선택에 얼마나 민감한지 확인하지만, 그 경우에도 누락 회차를 관측한 것은 아니다. 선택이 계수에 미친 편향의 방향과 크기는 현재 자료로 확정할 수 없다.""")
new_blocks=blocks[prior_block_count:];del blocks[prior_block_count:]
position=next(i for i,b in enumerate(blocks) if b['id']=='support_heading')
blocks[position:position]=new_blocks
md('sensitivity_heading',"""## 지표·공간·표본을 바꾼 결과도 함께 공개한다

아래 네 사양은 새 회귀를 보기 전에 고정했다. 중심점100%와 ID수 비교는 **같은 관측행·결과변수·고정효과**를 쓴다. 다만 '1+회차2배'와 '1+공연 ID 수2배'는 다른 노출 변화라서 계수 차이를 측정오차 감소량으로 해석할 수 없다.

90% 기준은 더 많은 상권을 포함하지만 미가용 후보를 뺀 예정회차의 부분 합계를 사용한다. 최단거리 기준은 공연장과 상권의 일부 경계가 가까운 연결을 포함하며, 기본 표본과 관측행도 달라진다. 이 두 비교의 결과 차이를 특정 변경 하나의 인과효과로 설명할 수는 없다. 표에는 유의한 사양만 골라 남기지 않았다.""")
table('scenarios','네 사양의 추정 수·표본·다중비교 결과',[
 ('scenario','사양','text'),('outcome','결과','text'),('estimated','추정 업종','number'),('rows','회귀 행','number'),('clusters','전체 상권','number'),
 ('raw_significant','p<0.05','number'),('bh10_significant','BH10<0.05','number'),('bh20_significant','BH20<0.05','number')],'scenario')
table('total_sensitivity','총매출의 고정 비교: 상대차이 [개별 95% CI]',[
 ('industry','업종','text'),('primary_centroid100','기본 예정회차','text'),('id_same_sample','동일 표본 공연 ID 수','text'),
 ('centroid90','가용90% 예정회차','text'),('min_distance100','최단거리 예정회차','text')],'industry')
md('spatial_heading',"""## 공간 상관을 고려한 표준오차는 참고용 민감도다

상권별 회귀 점수를 모든 기간에 걸쳐 합한 뒤, 두 상권 중심점 간 거리가 가까울수록 높은 가중치를 주는 가우시안 공분산을 계산했다. 가중치는 exp[−(거리/1,000m)²]로 고정했다. 회귀계수와 표본은 기본 분석과 동일하다.

표의 배율은 공간 표준오차/상권 군집 표준오차다. 1보다 크면 이 설정에서 불확실성이 커지고 작으면 줄어든다. 공간 상관이 항상 표준오차를 키우는 것은 아니다. 이 구간은 **정규 근사의 참고용 95% 구간**이며, 상권이 조밀한 두 지역에서 정확한 유한표본 추론을 보장하지 않는다. 공간 p값을 새로 만들거나 유의성을 결정하는 기준으로 쓰지 않는다.""")
table('spatial','고정1km 공간 가중 공분산 민감도',[
 ('industry','업종','text'),('outcome_label','결과','text'),('spatial_to_cluster_se_ratio','SE 배율','number'),
 ('spatial_ci','상대차이 참고구간','text')],'industry')

In [6]:
md('remaining',f"""## 측정을 개선했지만 인과효과를 확인한 연구는 아니다

예정회차는 실제 관객의 노출량을 대신하는 지표다. 등록된 일반 요일 일정을 과거 기간에 반복했고, 날짜별 휴연·예외·과거 변경 이력과 관객 수는 확인하지 못했다. 가용률 100%인 일정에도 이 한계가 남는다.

상권 중심점500m도 근사다. {len(data_summary['centroids_outside_polygon'])}개 상권의 기하학적 중심점은 자기 폴리곤 밖에 있으며, 넓고 불규칙한 상권의 매출 전체가 공연장500m 안에서 발생한 것은 아니다. 현재 시설 좌표·상권 경계를 과거 분기에 적용했다. 서로 다른 상권이 같은 공연을 공유하는 구조도 남는다.

고정효과는 상권·업종의 불변 특성과 업종별 공통 분기 변화를 제거한다. 하지만 관광·유동인구·날씨·상권별 행사·점포 변화처럼 지역별로 변하는 요인은 여전히 공연량과 매출에 함께 관련될 수 있다. 점포당 매출은 모집단 일치가 확인되지 않아 만들지 않았다.

기존 결과를 본 뒤 진단에 따라 수정한 분석이므로 **탐색적 연구**다. 새 회귀 전에 네 사양을 고정한 것은 추가적인 결과 선택을 제한하는 조치이지, 이미 본 데이터가 독립 확인 표본으로 바뀐다는 뜻은 아니다.""")
md('next_steps',"""## 졸업논문에는 현재 결과와 자료 한계를 함께 제시한다

현재 자료로는 예정 공연량과 분기 매출의 관련성을 검토하고, 지표 정의·공간 연결·일정 가용성에 따라 표본과 결과가 어떻게 달라지는지를 연구 결과로 제시할 수 있다. 유의하지 않은 결과를 '공연의 경제적 효과가 없다'로 바꾸거나, 특정 민감도 사양만 골라 결론내리면 안 된다.

추가 데이터 작업의 우선순위는 **기본 표본에서 빠진 공연 밀집 지역의 일정 검증**이다. 대학로·홍대의 누락 후보부터 날짜별 실제 회차·휴연·변경 이력을 확보하면, 표본을 넓히면서 노출의 불확실성을 줄일 수 있다. 파싱률만 올리는 것으로 실제 일정 검증을 대신하지 않는다.

공연 직후 소비를 연구하려면 더 좁은 공간의 일자·시간대 매출과 실제 공연 이력이 필요하다. 일별6구간 자료는 분기 자료보다 개선되지만 정확한 종료 후 3시간·6시간을 직접 측정하는 데는 여전히 제약이 있다. 자료 확보 전까지는 새로운 시간창을 반복 탐색하지 않는다.

남은 핵심 질문은 공연 밀집 지역의 일정을 검증해 포함했을 때 결과가 유지되는지, 실제 관객 수와 좁은 소비 영역을 관측하면 어느 정도의 관계를 정밀하게 추정할 수 있는지다.""")

## 4. 실제 결과에 근거한 해석과 계산 검증

통계적 유의성의 유무와 추정 정밀도를 구분한다. 사양별 결과를 모두 보존하며, 표본·단위가 다른 이전 계수와 크기 차이를 개선 효과로 해석하지 않는다.

In [7]:
assert main.status.eq('estimated').all() and effects.status.eq('estimated').all()
assert not effects.p_value.lt(.05).any()
assert analysis_summary['all_inputs_unchanged'] and analysis_summary['protected_unchanged']
assert frames['implementation'].passed.all() and selection_summary['all_inputs_unchanged']

def insert_after(target,key,body,source=None):
    md(key,body,source); item=blocks.pop(); pos=next(i for i,b in enumerate(blocks) if b['id']==target)
    blocks.insert(pos+1,item)

def result(industry,outcome=TOTAL,scenario=PRIMARY):
    return effects.loc[effects.industry.eq(industry)&effects.outcome.eq(outcome)&effects.scenario.eq(scenario)].iloc[0]

korean=result('한식음식점');chinese=result('중식음식점');fast=result('패스트푸드점');chicken=result('치킨전문점')
western_night=result('양식음식점',NIGHT)
insert_after('title','summary',f"""## 요약: 지표를 수정해도 뚜렷한 분기 매출 관련성은 확인되지 않았다

**예정회차·총매출액·야간 매출액으로 수정해 다시 분석했지만, 기본 20개 결과 모두 보정 전부터 p≥0.05였다.** 미리 정한 공연 ID 수·공간·가용률 비교까지 포함한 80개 결과에서도 p<0.05인 항목은 없었다. 여러 사양은 같은 자료를 반복 사용하므로 80개의 독립적인 재현 실험은 아니다.

한식 총매출의 상대차이는 {interval(korean)}, 패스트푸드는 {interval(fast)}다. 비교는 **1+예정회차2배에 대한 매출+1의 조건부 기하평균 차이**이며, 괄호는 개별 95% 신뢰구간이다. 한식은 추정치가0에 가깝고, 패스트푸드는 상당한 양의 관련성도 구간 안에 남는다. 야간 매출은 일부 업종에서 구간이 더 넓다.

**가장 중요한 한계는 일정 가용률 100% 규칙이 공연 밀집 상권을 많이 제외한다는 점**이다. 대학로·홍대입구역 관측은 기본 표본에 포함되지 않는다. 따라서 이번 결과는 '공연의 소비 효과가 없다'는 결론이 아니라, 정해진 공개자료·표본에서 뚜렷한 분기 관련성을 확인하지 못했다는 결과다. 보고서에는 전체 표본 흐름, 고정된 비교 사양, 대표성 제약과 재현 검증을 함께 남겼다.""")
insert_after('total_block','total_interpretation',f"""한식은 {interval(korean)}, 중식은 {interval(chinese)}로 모형상 비교의 점추정치가0근처다. 반면 패스트푸드는 {interval(fast)}, 치킨은 {interval(chicken)}다. 뒤의 두 업종은 크기가 더 크고 구간도 넓어 방향을 확정하기 어렵다.

이처럼 비유의성에는 **작게 추정된 관계**와 **불확실한 관계**가 함께 있다. 모두 '검정력이 부족해서 놓쳤다'고 단정할 수 없고, 반대로 모두 '실제로 관계가 없다'고 단정할 수도 없다. 경제적으로 무시할 수 있는 크기를 사전에 정한 동등성 검정도 수행하지 않았다.""",'total')
zero=frames['zero'].loc[frames['zero'].scenario.eq(PRIMARY)]
coffee_zero=float(zero.loc[zero.industry.eq('커피-음료'),'night_zero_pct'].iloc[0])
snack_zero=float(zero.loc[zero.industry.eq('분식전문점'),'night_zero_pct'].iloc[0])
insert_after('night_block','night_interpretation',f"""양식 야간 매출은 {interval(western_night)}다. 양의 점추정치만으로 야간 소비 증가가 확인됐다고 말할 수 없으며, 음의 관련성도 구간에 포함된다.

야간 매출이 실제0원으로 제공된 행은 커피·음료의 {coffee_zero:.2f}%, 분식의 {snack_zero:.2f}%다. 누락 매출을0으로 채운 결과가 아니다. ln(1+원) 모형에는0원과 양수 사이의 전환이 포함되고, **0이 포함된 로그+1 결과는 화폐 단위 선택의 영향도 받는다**. 야간 계수를 양의 매출만 있는 집단의 단순 성장률로 읽지 않으며, 표본을 임의로 제거해 유의성을 만들지 않는다.""")
insert_after('total_sensitivity_block','sensitivity_interpretation',"""예정회차90% 표본은6,791행·127상권으로 늘어나지만 모든 업종의 총매출·야간 매출 개별 구간이0을 포함한다. 기본 표본에서의 공연 ID 수 비교5,015행·114상권, 최단거리 기준3,537행·95상권도 마찬가지다. 따라서 이번 자료에서 비유의성이 BH보정 하나 때문에 발생했다거나, 회차 지표로 바꾸면 바로 사라진다고 볼 근거는 없다.

동일 표본에서 한식 총매출은 공연 ID 수 기준 −0.74%에서 예정회차 기준 +0.32%로 방향이 달라지고, 일식은 +5.33%에서 +0.19%로 달라진다. 두 지표가 같은 정보를 담지 않는다는 점은 확인되지만 어느 쪽이 실제 관객 노출의 정답인지는 확인하지 못했다. 이러한 비교로 실제 효과가0이라고 입증한 것은 아니다.""",'total_sensitivity')
insert_after('spatial_block','spatial_result',f"""이 설정에서 공간/군집 표준오차 비율은 {spatial.spatial_to_cluster_se_ratio.min():.2f}~{spatial.spatial_to_cluster_se_ratio.max():.2f}다. 공간 참고구간도 모두0을 포함한다. 항등 가중치(K=I)일 때 기존 군집 공분산을 재현하고, 가중행렬과 공분산의 양의 준정부호를 점검했다. 이는 구현 검증이며 미지의 실제 공간 상관 구조가 맞게 추정됐다는 보장은 아니다.""",'spatial')

info=frames['information'].loc[frames['information'].scenario.eq(PRIMARY)&frames['information'].outcome.eq(TOTAL)].copy()
info['top5_pct']=100*info.top5_information_share
dataset('information',info,paths['information'],'고정효과 제거 후 예정회차 설명변수의 제곱합을 상권별로 분해한 정보 집중도. 실제 관객 점유율·효과크기·유효표본수가 아님.')
common=frames['common10'].loc[frames['common10'].scenario.eq(PRIMARY)&frames['common10'].sample_stage.eq('estimated_industries_only')].iloc[0]
start=len(blocks)
md('information_heading',f"""## 공연량의 변화는 남아 있지만 일부 업종은 소수 상권에 집중된다

고정효과 후 ln(1+예정회차)의 잔차 제곱합은 원시 중심화 제곱합의 **{info.fe_variance_retained_pct.min():.2f}~{info.fe_variance_retained_pct.max():.2f}%**다. 공연량의 변화를 전부 제거한 모형은 아니다. 이 비율은 이전 분석과 표본·노출 정의가 다르므로 증가분을 측정 개선 효과로 해석하지 않는다.

아래는 추정에 쓰이는 설명변수 변동 중 상위5개 상권의 비중이다. 일식은 약61.4%, 치킨은 약52.1%로 소수 상권에 집중된다. 이 수치는 실제 매출·관객의 점유율이나 독립 표본수는 아니다. 정보가 어디에 집중됐는지 보여주는 진단이다.""",'information')
bar('information','상위5개 상권의 설명변수 변동 비중','industry','top5_pct','잔차화 노출 제곱합 비중 (%)','%','기본 분석 · 예정회차 · 고정효과 제거 후 업종별 제곱합 기준')
md('common_sample',f"""10종 매출이 같은 상권·분기에 모두 관측된 기본 표본은 {int(common.all10_areas)}상권·{int(common.all10_area_quarters)}상권분기, {int(common.all10_rows):,}행({common.all10_row_share_pct:.1f}%)뿐이다. 공통표본을 억지로 적용하면 최소20상권 조건을 충족하지 못한다. 업종 간 계수 차이에는 관측 지역 차이가 함께 포함될 수 있으므로, 현재 표를 업종의 순위나 반응 차이의 통계적 검정으로 읽지 않는다.""")
added=blocks[start:];del blocks[start:]
pos=next(i for i,b in enumerate(blocks) if b['id']=='remaining');blocks[pos:pos]=added
insert_after('method','cluster_detail',"""업종별 기울기를 하나의 결합 모형으로 추정하므로 G는 해당 사양의 전체 관측 상권 수다. 기본은 G=114, 자유도113이고 업종별 관측 상권은33~106개다. 군집 추론의 유한표본 한계는 남으며 업종별 상권 수보다 많은 독립 정보가 생긴 것은 아니다. [Cameron·Miller의 군집 추론 설명](https://cameron.econ.ucdavis.edu/research/Cameron_Miller_JHR_2015.pdf)에서 다루는 군집 간 독립 가정 때문에 공간 민감도를 별도로 보고했다.""")
insert_after('remaining','validation',f"""원자료와 기존 기준선은 보존했다. 수정 데이터의 원 매출값·결측을 대조하고, 축소한 실제 표본의 명시적 고정효과 OLS와 잔차화 계산을 비교했다. 구현 점검 {analysis_summary['implementation_checks_passed']}개가 통과했으며 계수의 최대 차이는 약2.57×10⁻¹⁴였다. 같은 표본의 예정회차/ID비교도 관측키 해시로 확인했다. 계산 재현성과 데이터 정확성·인과성은 별개의 검증이다.""",'analysis_summary')
print('주결과',len(main),'행 / 전체',len(effects),'행 / 본문',len(blocks),'블록')

주결과 20 행 / 전체 80 행 / 본문 30 블록


## 5. 하나의 읽기용 HTML로 저장

표·차트 출처에 실행된 SQL과 원본 CSV를 기록한다. 동일 artifact로 HTML을 패키징하고 구조 검증을 수행한다. 별도의 HTML 구현을 유지하지 않는다.

In [8]:
generated=datetime.now(ZoneInfo('Asia/Seoul')).isoformat()
artifact={'surface':'report','manifest':{'version':1,'surface':'report','title':title,
 'generatedAt':generated,'description':'진단에 따라 예정회차·매출액·공간 연결·표본 기준을 수정하고 재실행한 졸업 프로젝트 결과',
 'blocks':blocks,'tables':tables,'charts':charts,'sources':sources},
 'snapshot':{'version':1,'generatedAt':generated,'status':'ready','datasets':datasets,'accessIssues':[]},'sources':sources}
assert all(sha(ROOT/path)==value for path,value in input_hashes.items())
assert len({b['id'] for b in blocks})==len(blocks)
(OUT/'artifact.json').write_text(json.dumps(artifact,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
notes={'audience':'technical','delivery':'portable_html','question':plan['question'],
 'exploratory_after_prior_results':True,'new_plan_frozen_before_fitting':plan['frozen_before_revised_model_fitting'],
 'source_sha256':input_hashes,'baselines_modified':False,'p_based_specification_selection':False,
 'required_structure_mapping':{'title':'title','technical_summary':'summary','definitions':'definitions/changes',
 'findings':'selection_heading/support_heading/total_heading/night_heading','methodology':'method/cluster_detail',
 'uncertainty':'sensitivity_heading/spatial_heading/information_heading/remaining','next_steps':'next_steps','further_questions':'next_steps'},
 'structure_change_reason':'측정 정의와 표본 대표성을 계수표보다 먼저 보여 해석 범위를 명확히 함',
 'chart_map':[{'id':'support','type':'horizontalBar','claim':'업종별 관측 지역이 다름'},
              {'id':'density','type':'horizontalBar','claim':'완전 일정 규칙이 공연 밀집 관측을 더 많이 제외함'},
              {'id':'information','type':'horizontalBar','claim':'일부 업종의 노출 변동이 소수 상권에 집중됨'}],
 'table_rationale':'주결과·민감도는 효과크기와개별 95%CI 및 BH10/BH20를 함께 정확히 조회해야 하므로표사용; 점추정치 단독 막대그래프를 사용하지 않음',
 'effect_plot':'reports/final/revised_quarterly/13_primary_effects.png (분석 노트북의 정적 CI 그림)',
 'sharing_assessment':'Share with caveats; complete-ID selection excludes dense core areas',
 'key_validation':{'model_checks':analysis_summary['implementation_checks_passed'],'primary_raw_significant':int(main.p_value.lt(.05).sum()),'all80_raw_significant':int(effects.p_value.lt(.05).sum())}}
(OUT/'report_notes.json').write_text(json.dumps(notes,ensure_ascii=False,indent=2),encoding='utf-8')
print('artifact 저장: 표',len(tables),'차트',len(charts),'근거자료',len(datasets))

artifact 저장: 표 5 차트 3 근거자료 8


In [9]:
builder=Path('C:/Users/mamap/.codex/plugins/cache/openai-curated-remote/data-analytics/0.2.10-13ceeea1f599/skills/build-report/scripts/deliver_portable_artifact.mjs')
result_build=subprocess.run([shutil.which('node'),str(builder),'--input',str(OUT/'artifact.json'),'--output',str(OUT/'report.html')],
    capture_output=True,text=True,encoding='utf-8',timeout=120)
print(result_build.stdout[-6000:])
if result_build.returncode:print(result_build.stderr[-6000:])
assert result_build.returncode==0
(OUT/'delivery_receipt.json').write_text(result_build.stdout,encoding='utf-8')
receipt=json.loads(result_build.stdout.strip())
notes['delivery_verification']=receipt['stages']['verification']
notes['browser_visual_qa_performed']=notes['delivery_verification']=='passed'
notes['qa_limitation']='브라우저 실행 환경 부재로 화면 자동검사 미수행. 데이터·페이로드 일치·문서구조 검증 통과.' if not notes['browser_visual_qa_performed'] else ''
(OUT/'report_notes.json').write_text(json.dumps(notes,ensure_ascii=False,indent=2),encoding='utf-8')
assert all(sha(ROOT/path)==value for path,value in input_hashes.items())
assert all(sha(ROOT/path)==value for path,value in plan['protected_source_sha256'].items())
print('보고서 완료 / 원본·기존 결과 불변')

{"ok":true,"html":"C:\\Coding\\졸업프로젝트\\reports\\final\\revised_quarterly\\report.html","stages":{"validation":"passed","package":"passed","verification":"structural_only"},"browserWarning":{"code":"browser_unavailable","message":"No installed Chromium headless-shell executable was found. Set CHROMIUM_EXECUTABLE_PATH (or the legacy PLAYWRIGHT_EXECUTABLE_PATH) or preinstall Chromium headless-shell under PLAYWRIGHT_BROWSERS_PATH; portable tooling never downloads browsers."},"counts":{"blocks":30,"charts":3,"html":0,"metrics":0,"tables":5},"sourceDialog":"not_verified","sourceInteraction":"not_verified","viewports":[],"timings":{"validateAndPackageMs":136.6,"staticChartExtractionMs":49.1,"structuralVerificationMs":5.9,"totalMs":151.2}}

보고서 완료 / 원본·기존 결과 불변
